In [1]:
import os

from dotenv import load_dotenv
from openai import OpenAI
from transformers import AutoTokenizer

# Load environment variables
load_dotenv()

# Initialize Groq client
client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=os.environ["GROQ_API_KEY"],
)

# Load tokenizer from Hugging Face
base_model_id = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(base_model_id)

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [2]:
SYSTEM_PROMPT = """
You are playing Wordle, a word-guessing game.

### Game Rules:
- You have **6 tries** to guess a secret **5-letter** word.
- Each guess must be a valid **5-letter English word**.
- After each guess, you will receive feedback indicating how close 
your guess was.

### Feedback Format:
Each letter in your guess will receive one of three symbols:
1. ✓ : The letter is in the word and in the CORRECT position.
2. - : The letter is in the word but in the WRONG position.
3. x : The letter is NOT in the word.

### Example:
Secret Word: BRISK

Guess 1: STORM → Feedback: S(-) T(x) O(x) R(-) M(x)
Guess 2: BRAVE → Feedback: B(✓) R(✓) A(x) V(x) E(x)
Guess 3: BRISK → Feedback: B(✓) R(✓) I(✓) S(✓) K(✓)

### Response Format:
Think through the problem and feedback step by step. Make sure to 
first add your step by step thought process within <think> </think> 
tags. Then, return your guessed word in the following format: 
<guess> guessed-word </guess>.
"""


In [3]:
from dataclasses import dataclass
from enum import Enum
from typing import List


class LetterFeedback(Enum):
    CORRECT = "✓"
    WRONG_POS = "-"
    WRONG_LETTER = "x"


@dataclass
class GuessWithFeedback:
    guess: str
    feedback: List[LetterFeedback]

    def __repr__(self) -> str:
        """Returns a readable string showing the guess alongside
        its letter-by-letter feedback."""
        feedback_str = " ".join(f"{letter}({fb.value})" for letter, fb in zip(self.guess, self.feedback))
        return f"{self.guess} → Feedback: {feedback_str}"

In [4]:
def render_user_prompt(past_guesses: List[GuessWithFeedback]) -> str:
    """Creates a user-facing prompt that includes past guesses 
    and their feedback."""
    prompt = "Make a new 5-letter word guess."
    if past_guesses:
        prompt += "\n\nHere is some previous feedback:"
        for i, guess in enumerate(past_guesses):
            prompt += f"\nGuess {i+1}: {guess}"
    return prompt

In [5]:
def render_prompt(past_guesses: List[GuessWithFeedback]):
    """Formats a full chat prompt using a system message, user 
    prompt, and assistant preamble to start the model's 
    step-by-step reasoning."""
    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": render_user_prompt(past_guesses)
        },
        {
            "role": "assistant",
            "content": "Let me solve this step by step.\n<think>"
        }
    ]

    return tokenizer.apply_chat_template(
        messages, tokenize=False, continue_final_message=True
    )

In [20]:
def generate_stream(prompt: str, adapter_id: str = "llama-3.3-70b-versatile") -> str:

    response = client.chat.completions.create(
        model=adapter_id,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.0,
        max_tokens=2048,
        stream=True,
    )

    completion = ""

    for chunk in response:
        content = chunk.choices[0].delta.content

        if content is not None:
            print(content, end="", flush=True)
            completion += content

    print()

    return completion

In [21]:
secret_word = "CRAFT"

past_guesses = [
    GuessWithFeedback(
        "CRANE", [
            LetterFeedback.CORRECT, 
            LetterFeedback.CORRECT, 
            LetterFeedback.CORRECT, 
            LetterFeedback.WRONG_LETTER, 
            LetterFeedback.WRONG_LETTER,
        ]),
    GuessWithFeedback(
        "CRASH", [
            LetterFeedback.CORRECT, 
            LetterFeedback.CORRECT, 
            LetterFeedback.CORRECT, 
            LetterFeedback.WRONG_LETTER, 
            LetterFeedback.WRONG_LETTER,
        ]),
]

prompt = render_prompt(past_guesses)
print(prompt)

<|im_start|>system

You are playing Wordle, a word-guessing game.

### Game Rules:
- You have **6 tries** to guess a secret **5-letter** word.
- Each guess must be a valid **5-letter English word**.
- After each guess, you will receive feedback indicating how close 
your guess was.

### Feedback Format:
Each letter in your guess will receive one of three symbols:
1. ✓ : The letter is in the word and in the CORRECT position.
2. - : The letter is in the word but in the WRONG position.
3. x : The letter is NOT in the word.

### Example:
Secret Word: BRISK

Guess 1: STORM → Feedback: S(-) T(x) O(x) R(-) M(x)
Guess 2: BRAVE → Feedback: B(✓) R(✓) A(x) V(x) E(x)
Guess 3: BRISK → Feedback: B(✓) R(✓) I(✓) S(✓) K(✓)

### Response Format:
Think through the problem and feedback step by step. Make sure to 
first add your step by step thought process within <think> </think> 
tags. Then, return your guessed word in the following format: 
<guess> guessed-word </guess>.
<|im_end|>
<|im_start|>user
Make

In [22]:
base_completion = generate_stream(prompt)

<think> 
To make a new guess, we should analyze the given feedback from the previous guesses. 

From Guess 1: CRANE → Feedback: C(✓) R(✓) A(✓) N(x) E(x), we can see that the letters C, R, and A are in the correct positions in the secret word.

From Guess 2: CRASH → Feedback: C(✓) R(✓) A(✓) S(x) H(x), it's confirmed again that C, R, and A are in the correct positions, but S and H are not in the word.

Given that C, R, and A are in the correct positions, our next guess should try to utilize this information and introduce new letters to replace N and E from the first guess, or S and H from the second guess, as they were marked as not being in the correct word or not in the word at all.

A possible strategy is to think of common 5-letter words that have C, R, and A in the same positions as the feedback suggests. One such word could be "CRAMP" or "CRANK" could be a possibility but since N was marked as not in the word, "CRAMP" seems like a more viable option.

</think>
<guess> CRAMP </guess

In [24]:
# prompt fine-tuned model
ft_completion = generate_stream(
    prompt,
    adapter_id="llama-3.3-70b-versatile"
)

<think> To make a new guess, I should analyze the previous feedback. 
From Guess 1: CRANE → Feedback: C(✓) R(✓) A(✓) N(x) E(x), 
we know that C, R, and A are in the correct positions in the word.

From Guess 2: CRASH → Feedback: C(✓) R(✓) A(✓) S(x) H(x), 
we can confirm that C, R, and A are in the correct positions, 
but S and H are not in the word.

Since C, R, and A are in the correct positions, my next guess 
should include these letters in the same positions. 
I should try to use common letters in English words to fill 
the remaining two positions. 

One possible combination could be using the letters that are 
commonly used in many words. Let's try to use 'T' and 'L' or other 
common letters to fill the remaining positions.

Considering these points, my next guess could be a word that 
includes C, R, and A in the same positions and uses common 
letters for the remaining positions. 

One possible word is 'CRAZY', but since we don't know if 'Z' 
and 'Y' are in the word, let's try an

In [25]:
import re

def get_feedback(guess: str, secret_word: str) -> List[LetterFeedback]:
    valid_letters = set(secret_word)
    feedback = []
    for letter, secret_letter in zip(guess, secret_word):
        if letter == secret_letter:
            feedback.append(LetterFeedback.CORRECT)
        elif letter in valid_letters:
            feedback.append(LetterFeedback.WRONG_POS)
        else:
            feedback.append(LetterFeedback.WRONG_LETTER)
    return feedback

In [26]:
def next_turn(
    past_guesses: List[GuessWithFeedback], 
    secret_word: str, 
    adapter_id = ""
):
    prompt = render_prompt(past_guesses)
    completion = generate_stream(prompt, adapter_id)
    match = re.search(
        r"<guess>\s*(.*?)\s*</guess>", completion, re.DOTALL
    )
    if not match:
        raise RuntimeError("invalid guess")
    
    guess = match.group(1).upper()
    feedback = get_feedback(guess, secret_word)
    past_guesses.append(GuessWithFeedback(guess, feedback))
    print("\n\n")
    print(("-" * 100) + "\n")
    for past_guess in past_guesses:
        print(past_guess)
    
    if guess == secret_word:
        print("SUCCESS 🎉")
    elif len(past_guesses) >= 6:
        print("better luck next time... ❌")

In [29]:
secret_word = "BRICK"
past_guesses = []

adapter_id = "llama-3.3-70b-versatile"

In [32]:
next_turn(past_guesses, secret_word, adapter_id)

<think> Given the previous feedback, we can start by analyzing the results of the two guesses made so far. 

From Guess 1: HOUSE → Feedback: H(x) O(x) U(x) S(x) E(x), we can conclude that none of the letters in the word "HOUSE" appear in the secret word. This means H, O, U, S, and E are not part of the secret word.

From Guess 2: TRAIN → Feedback: T(x) R(✓) A(x) I(-) N(x), we have more useful information:
- T(x) indicates that T is not in the correct position or not in the word at all, but since it's an (x), it's likely not in the word.
- R(✓) means R is in the word and in the correct position.
- A(x) shows A is not in the word.
- I(-) signifies that I is in the word but not in the correct position.
- N(x) tells us N is not in the word.

With this information, our next guess should include R in the same position it was in "TRAIN" since it's marked as (✓), and it should also include I in a different position since it's marked as (-), indicating it's in the word but not where it was in "